In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision import models

import wandb

import sys
sys.path.append('../../../../src/utils/')
sys.path.append('../../../../src/benchmark/')
from build_model import resnet50_, resnet34_, densenet161_, fpn_resnet50_classification, xcit_small, xcit_medium
from train_functions import train_epochs
from utils import hdf5_dataset, list_to_dict, viz_dataloader, split_train_valid
from RotateConvolution import RotateConv2d

In [2]:
# with h5py.File('/mnt/raid0/yichen/imagenet_atom_noise_v4_rot_10m_100k_subset_transform.h5', 'r') as h5:
#     viz_h5_structure(h5)
num_workers = 8
batch_size = 15000
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)

# imagenet
imagenet_ds = hdf5_dataset('/mnt/raid0/yichen/imagenet_v4_rot_10m_train_unchunked.h5', folder='train', transform=transforms.ToTensor())
train_ds, valid_ds = split_train_valid(imagenet_ds, train_ratio=0.8, seed=42)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# viz_dataloader(train_dl, label_converter=label_converter, title='imagenet-train-10m')

valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# viz_dataloader(valid_dl, label_converter=label_converter, title='imagenet-valid-2m')

# atom-cv-2m
cv_atom_ds_2m = hdf5_dataset('../../../../datasets/atom_v4_rot_2m_unchunked.h5', folder='test', transform=transforms.ToTensor())
cv_atom_dl_2m = DataLoader(cv_atom_ds_2m, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# viz_dataloader(cv_atom_dl_2m, label_converter=label_converter, title='atom-cv-2m')

# noise-cv-2m
cv_noise_ds_2m = hdf5_dataset('../../../../datasets/noise_v4_rot_2m-test.h5', folder='test', transform=transforms.ToTensor())
cv_noise_dl_2m = DataLoader(cv_noise_ds_2m, batch_size=batch_size, shuffle=True, num_workers=num_workers)
# viz_dataloader(cv_noise_dl_2m, label_converter=label_converter, title='noise-cv-2m')

In [3]:
class multikernel_resnet34_(nn.Module):
    def __init__(self, in_channels, n_classes, dropout, kernel_sizes):
        super(multikernel_resnet34_, self).__init__()
        self.model_list = nn.ModuleList()
        for kernel_size in kernel_sizes:
            model = resnet34_(in_channels=in_channels, n_classes=n_classes, dropout=dropout)

            model.conv1 = nn.Sequential(
                                RotateConv2d(in_channels, 12, kernel_size=kernel_size, padding=int(kernel_size/2), stride=4, trainable=False),
                                nn.Conv2d(12, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
                                         )

            model.fc = nn.Identity()  # Remove the final fully connected layer
            self.model_list.append(model)

        self.classifier = nn.Sequential(nn.BatchNorm1d(2048),
                                        nn.Dropout(p=dropout, inplace=False),
                                        nn.Linear(in_features=2048, out_features=512, bias=False),
                                        nn.ReLU(inplace=True),

                                        nn.BatchNorm1d(512),
                                        nn.Dropout(p=dropout, inplace=False),
                                        nn.Linear(in_features=512, out_features=64, bias=False),
                                        nn.ReLU(inplace=True),
                                        
                                        nn.BatchNorm1d(64),
                                        nn.Dropout(p=dropout, inplace=False),
                                        nn.Linear(in_features=64, out_features=n_classes, bias=True)
                                        )

    def forward(self, x):
        fv_list = [model(x) for model in self.model_list]
        # for f in fv_list:
            # print(f.shape)
        fv = torch.cat(fv_list, dim=1)
        # print(fv.shape)
        return self.classifier(fv)

model = multikernel_resnet34_(in_channels=3, n_classes=17, dropout=0.5, kernel_sizes=[64, 48, 32, 16])
outputs = model(torch.randn(2,3,256,256))
outputs.shape

torch.Size([2, 17])

In [4]:
model = torch.load('../../../../saved_models/ResNet34_MultiKernel_rot_conv2d_fixed-10m/epoch-9.pt').module
model = model.to(torch.device('cpu'))
outputs = model(torch.randn(2,3,256,256))
print(outputs.shape)
model = torch.nn.DataParallel(model, device_ids=[5,6,7,8,9])
device = torch.device('cuda:5')

torch.Size([2, 17])


In [5]:
wandb.login()
config = {
    'dataset': '10 million datasets',
    'loss_func': 'CrossEntropyLoss', # nn.MSELoss()
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
}

NAME = 'ResNet34_MultiKernel_rot_conv2d_fixed-10m'

proj_name = 'Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning'
wandb.init(project=proj_name, entity='yig319', name=NAME, id=NAME, group='preprocess', save_code=True, config=config)
config = wandb.config

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yig319. Use `wandb login --relogin` to force relogin


In [8]:
lr = 1e-3
start = 0
epochs = 20

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
                            

history = train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl, 
                       cv_dl_list=[cv_atom_dl_2m, cv_noise_dl_2m], cv_name_list=['cv_atom_2m', 'cv_noise_2m'],
                       epochs=epochs, start=start, scheduler=scheduler, model_dir=f'../../../../saved_models/{NAME}/', tracking=True)

Epoch: 1/20


100%|██████████| 3201/3201 [6:32:28<00:00,  7.36s/it]  


Training: Loss: 1.5037, Accuracy: 50.5271%.


100%|██████████| 801/801 [1:33:04<00:00,  6.97s/it]


Validation: Loss: 0.8670, Accuracy: 69.5806%.


...

Training: Loss: 0.3572, Accuracy: 90.2073%.


100%|██████████| 801/801 [1:32:16<00:00,  6.91s/it]


Validation: Loss: 1.4606, Accuracy: 59.9529%.


 25%|██▌       | 207/814 [23:47<1:09:47,  6.90s/it]


KeyboardInterrupt: 

In [6]:
lr = 1e-3
start = 10
epochs = 20

loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, epochs=epochs, max_lr=lr, 
                            steps_per_epoch=len(train_dl))
                            

history = train_epochs(model, loss_func, optimizer, device, train_dl, valid_dl, 
                       cv_dl_list=[cv_atom_dl_2m, cv_noise_dl_2m], cv_name_list=['cv_atom_2m', 'cv_noise_2m'],
                       epochs=epochs, start=start, scheduler=scheduler, model_dir=f'../../../../saved_models/{NAME}/', tracking=True)

Epoch: 11/30


  0%|          | 0/534 [00:00<?, ?it/s]

100%|██████████| 534/534 [3:53:21<00:00, 26.22s/it]  


Training: Loss: 0.2614, Accuracy: 93.4224%.


100%|██████████| 134/134 [57:13<00:00, 25.63s/it]


...

100%|██████████| 136/136 [56:54<00:00, 25.11s/it]


cv_atom_2m: Loss: 3.2774, Accuracy: 28.9431%.


100%|██████████| 134/134 [55:40<00:00, 24.93s/it]


cv_noise_2m: Loss: 3.4727, Accuracy: 9.4714%.
Epoch: 28/30


 48%|████▊     | 256/534 [1:50:32<1:58:39, 25.61s/it]